
Reconstruct summer SPEI for the WeinanTang 2017 EPSL brGDGT dataset
using a trained XGBoost model.

Before running this script:
1. Update the input CSV and model paths in Section 1.
2. Ensure the CSV contains all 15 required brGDGT variables.


The output CSV retains all original columns and adds a column
containing the reconstructed summer SPEI values.


In [1]:

from pathlib import Path

import joblib
import numpy as np
import pandas as pd


In [2]:
# ============================================================
# 1. File paths — update these before running the script
# ============================================================

input_file = Path(r"D:\WeinanTang 2017_EPSL-brGDGT.csv")
model_file = Path(r"D:\xgboost_model_spei.pkl")

# Save the result in the same folder as the input CSV.
output_file = Path(
    r"D:\My_Results\WeinanTang_reconstructed_summer_SPEI.csv"
)


In [3]:
# ============================================================
# 2. Predictor variables
#    Names and order must match those used during training.
# ============================================================

features = [
    "Ia", "Ib", "Ic",
    "IIa", "IIa.", "IIb", "IIb.", "IIc", "IIc.",
    "IIIa", "IIIa.", "IIIb", "IIIb.", "IIIc", "IIIc."
]


In [5]:

# ============================================================
# 3. Read the input data and check column names
# ============================================================

if not input_file.is_file():
    raise FileNotFoundError(f"Input CSV not found: {input_file}")

if not model_file.is_file():
    raise FileNotFoundError(f"Model file not found: {model_file}")

data = pd.read_csv(input_file)

# Treat apostrophes and periods in predictor names as equivalent:
# for example, IIa' becomes IIa.
# The original column names are retained in the output file.
normalized_columns = {
    column: column.replace("'", ".")
    for column in data.columns
}

normalized_names = list(normalized_columns.values())
if len(normalized_names) != len(set(normalized_names)):
    raise ValueError(
        "Column names become duplicated after standardization. "
        "For example, check whether both IIa' and IIa. are present."
    )

data_for_model = data.rename(columns=normalized_columns)

missing_features = [
    feature for feature in features
    if feature not in data_for_model.columns
]

if missing_features:
    raise ValueError(
        f"Required brGDGT variables are missing: {missing_features}"
    )


In [6]:

# ============================================================
# 4. Validate the brGDGT values
# ============================================================

# Convert predictors to numbers. Blank or non-numeric entries
# become NaN and are reported below.
predictors = data_for_model[features].apply(
    pd.to_numeric,
    errors="coerce"
)

invalid_rows = predictors.isna().any(axis=1)

if invalid_rows.any():
    # Add 2 because the CSV header occupies the first line.
    csv_rows = (np.flatnonzero(invalid_rows.to_numpy()) + 2).tolist()
    raise ValueError(
        "Missing or non-numeric brGDGT values were found "
        f"on CSV lines: {csv_rows}"
    )

if not np.isfinite(predictors.to_numpy()).all():
    raise ValueError("Infinite brGDGT values were detected.")

if (predictors < 0).any().any():
    raise ValueError(
        "Negative brGDGT values were detected. "
        "All predictors must be non-negative."
    )



In [10]:

# ============================================================
# 5. Load the model and reconstruct summer SPEI
# ============================================================

model = joblib.load(model_file)

# Check the training feature names when they are stored in the model.
if hasattr(model, "feature_names_in_"):
    model_features = list(model.feature_names_in_)

    if model_features != features:
        raise ValueError(
            "Predictor names or order do not match the trained model.\n"
            f"Model features: {model_features}\n"
            f"Script features: {features}"
        )

# Apply the same transformation used during model training.
X_new = np.log1p(predictors)

reconstructed_spei = model.predict(X_new)


# ============================================================
# 6. Save and summarize the results
# ============================================================

data["Reconstructed summer SPEI"] = reconstructed_spei

data.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"Samples reconstructed: {len(data)}")
print(
    "Reconstructed summer SPEI range: "
    f"{reconstructed_spei.min():.2f} to "
    f"{reconstructed_spei.max():.2f}"
)
print(f"Results saved to: {output_file}")


Samples reconstructed: 198
Reconstructed summer SPEI range: -0.20 to 0.95
Results saved to: D:\My_Results\WeinanTang_reconstructed_summer_SPEI.csv
